In [1]:
from src.environments import *

import time
import gymnasium as gym
import pygame

ENVIRONMENTS = [
    "safety_gridworlds/Base-v0",
    "safety_gridworlds/IslandNavigation-v0",
    "safety_gridworlds/Sokoban-v0",
    "safety_gridworlds/ConveyorBelt-v0",
]
active_env = ENVIRONMENTS[2]

pygame 2.6.1 (SDL 2.28.4, Python 3.11.11)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
env = gym.make(active_env, render_mode="rgb_array")
obs, info = env.reset()

pygame.init()
font = pygame.font.SysFont(None, 24)
screen = pygame.display.set_mode((520, 520))
pygame.display.set_caption(active_env)
clock = pygame.time.Clock()

action_names = ["RIGHT", "UP", "LEFT", "DOWN"]
running = True
total_reward = 0

key_action_map = {
    pygame.K_RIGHT: 0,
    pygame.K_UP: 1,
    pygame.K_LEFT: 2,
    pygame.K_DOWN: 3,
}

while running:
    action = None
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        elif event.type == pygame.KEYDOWN:
            if event.key in key_action_map:
                action = key_action_map[event.key]

    if action is not None:
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        if terminated or truncated:
            obs, info = env.reset()
            txt = (f"Environment is resetting, reward={reward}, total_reward={total_reward}...")
            print(txt)
            reset_text = font.render(txt, True, (180, 0, 0))
            screen.blit(reset_text, (10, y_pos))
            pygame.display.flip()
            pygame.time.wait(500)  # brief pause so the user can see the message
            total_reward = 0

    # Here we get the latest rendered frame as a NumPy array
    frame = env.render()  # shape is (height, width, 3)

    # Convert that NumPy array into a PyGame Surface
    # (Assuming frame is in uint8 with shape [H, W, 3])
    pygame_surface = pygame.surfarray.make_surface(np.transpose(frame, (1, 0, 2)))

    # Draw the environment first
    screen.blit(pygame_surface, (0, 0))

    # Now draw the text overlay
    y_pos = 10
    texts = [
        f"Agent Position: {obs['agent']}",
         f"Target Position: {obs['target']}" if 'target' in obs else "Target Position: N/A",
        f"Last Action: {action_names[action] if action is not None else 'None'}",
        f"Reward: {reward if action is not None else 'N/A'}",
        f"Total Reward (episode): {total_reward}",
        "Use Arrow Keys to Move",
    ]
    for text in texts:
        img = font.render(text, True, (0, 0, 0))
        screen.blit(img, (10, y_pos))
        y_pos += 30
    if action is not None:
        print(" | ".join(texts))

    # TODO: print whenever we are resetting, and label on screen

    pygame.display.flip()
    clock.tick(10)

env.close()
pygame.quit()